# RAG with Cohere Reranking

1. **Dense Retrieval (Vector Search):**  It might rank a partially relevant document higher simply because of similar keywords.
2. **Reranker (Cohere):**  It takes the top `K` documents retrieved by vector search and re-evaluates them using a cross-encoder model to score and sort the most semantically relevant context to the very top.


In [1]:
import os
import cohere
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_groq import ChatGroq

load_dotenv()

cohere_api_key = os.getenv("COHERE_API_KEY")


/Users/krishrohilla/Desktop/MindzKonnected/Tutorial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Mock Document Database


In [2]:
documents = [
    "AI agents are software programs that can perceive their environment, make decisions, and take actions using tools to achieve goals.",
    "RAG (Retrieval-Augmented Generation) improves LLM responses by fetching relevant documents from a database and placing them in the prompt.",
    "Making pizza at home requires a hot oven, fresh dough, tomato sauce, mozzarella cheese, and toppings like pepperoni or basil.",
    "Vector embeddings convert text into lists of numbers (vectors) that capture semantic meaning, allowing us to perform similarity searches.",
    "Cohere Rerank is a tool that takes a query and a list of documents and re-orders them from most relevant to least relevant.",
    "Climate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities like burning fossil fuels."
]

print(f"Loaded {len(documents)} documents in the database.")

Loaded 6 documents in the database.


### Initial Vector Search 


In [3]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

query = "What is an AI agent and how does it use tools?"

query_embedding = embedder.encode([query])
doc_embeddings = embedder.encode(documents)

similarities = cosine_similarity(query_embedding, doc_embeddings)[0]


retrieved_indices = np.argsort(similarities)[::-1][:4]


retrieved_docs = []
for idx in retrieved_indices:
    doc_text = documents[idx]
    score = similarities[idx]
    retrieved_docs.append(doc_text)
    print(f"Similarity Score: {score:.4f} | {doc_text}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5783.93it/s]


Similarity Score: 0.8779 | AI agents are software programs that can perceive their environment, make decisions, and take actions using tools to achieve goals.
Similarity Score: 0.1824 | Vector embeddings convert text into lists of numbers (vectors) that capture semantic meaning, allowing us to perform similarity searches.
Similarity Score: 0.1544 | Cohere Rerank is a tool that takes a query and a list of documents and re-orders them from most relevant to least relevant.
Similarity Score: 0.1423 | RAG (Retrieval-Augmented Generation) improves LLM responses by fetching relevant documents from a database and placing them in the prompt.


### Cohere Reranking


In [4]:
    co = cohere.ClientV2(api_key=cohere_api_key)
    
    response = co.rerank(
        model="rerank-english-v3.0",
        query=query,
        documents=retrieved_docs,
        top_n=3  
    )
    
    reranked_docs = []
    for rank in response.results:
        idx = rank.index
        score = rank.relevance_score
        doc_text = retrieved_docs[idx]
        reranked_docs.append(doc_text)
        print(f"Rerank Relevance Score: {score:.4f} | {doc_text}")

Rerank Relevance Score: 0.9997 | AI agents are software programs that can perceive their environment, make decisions, and take actions using tools to achieve goals.
Rerank Relevance Score: 0.0000 | RAG (Retrieval-Augmented Generation) improves LLM responses by fetching relevant documents from a database and placing them in the prompt.
Rerank Relevance Score: 0.0000 | Cohere Rerank is a tool that takes a query and a list of documents and re-orders them from most relevant to least relevant.


### RAG Generation
Send reranked context into prompt and send it to Groq's LLM 

In [5]:

final_context = reranked_docs if cohere_api_key else retrieved_docs
context_text = "\n".join([f"- {doc}" for doc in final_context])

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key, model_name="qwen/qwen3-32b", temperature=0.1)

prompt = f"""Use the following retrieved context to answer the user's question accurately.

Context:
{context_text}

Question: {query}
Answer:"""

response = llm.invoke(prompt)

print("Generated RAG Response")
print(response.content)

Generated RAG Response
<think>
Okay, the user is asking about AI agents and how they use tools. Let me start by recalling the context provided.

First, the context defines AI agents as software programs that can perceive their environment, make decisions, and take actions using tools to achieve goals. So I need to make sure I include that definition. Then, there's mention of specific tools like RAG and Cohere Rerank. 

The user wants to know how AI agents use tools. The example given is RAG, which fetches documents and places them in the prompt to improve LLM responses. Cohere Rerank is another tool that reorders documents by relevance. 

I should structure the answer by first explaining what an AI agent is, then discuss the role of tools in their operation. Use the examples from the context to illustrate how tools are utilized. Make sure to connect the tools to the agent's ability to make decisions and take actions. Also, check if there's any other information in the context that's re